# Corpus regression dataset

In [ ]:
from __future__ import annotations

import itertools

import plotly.express as px
import polars as pl
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from src import get_repo_base
from src.config.base import BaseConfig

dataset = "HuggingFaceFW/fineweb-edu"
dataset_config = "sample-10BT"
# rows pulled for distribution plots; separate from the collected corpus
eda_sample = 5_000
seed = 42

In [ ]:
class CorpusRegressionConfig(BaseConfig):
    sample_length: int  # canonical 128
    num_samples: int  # canonical 50_000 (per split; train and val each get this many)
    pretrained_tokenizer_model_name: str  # canonical 'HuggingFaceTB/SmolLM2-135M'

    @classmethod
    def canonical_kwargs(cls) -> dict[str, Any]:
        return {
            "sample_length": 128,
            "num_samples": 50_000,
            "pretrained_tokenizer_model_name": "HuggingFaceTB/SmolLM2-135M",
            "num_regression_bins": 8,
        }


cfg = CorpusRegressionConfig(
    sample_length=128,
    num_samples=50_000,
    pretrained_tokenizer_model_name="HuggingFaceTB/SmolLM2-135M",
    num_regression_bins=8,
)

tokenizer_slug = (
    cfg.pretrained_tokenizer_model_name.split("/")[-1].lower().replace("-", "_")
)
out_dir = (
    get_repo_base()
    / "artifacts"
    / "corpus-regression"
    / (f"fineweb_edu_{tokenizer_slug}_{cfg.num_samples}_x_{cfg.sample_length}")
)
out_dir.mkdir(parents=True, exist_ok=True)
out_train = out_dir / "train.parquet"
out_val = out_dir / "val.parquet"
cfg, out_train, out_val

(CorpusRegressionConfig(sample_length=128, num_samples=50000, pretrained_tokenizer_model_name='HuggingFaceTB/SmolLM2-135M', num_regression_bins=8),
 PosixPath('/home/nlyu/Code/maxrl-statistics/artifacts/corpus-regression/fineweb_edu_smollm2_135m_50000_x_128/train.parquet'),
 PosixPath('/home/nlyu/Code/maxrl-statistics/artifacts/corpus-regression/fineweb_edu_smollm2_135m_50000_x_128/val.parquet'))

In [3]:
tok = AutoTokenizer.from_pretrained(cfg.pretrained_tokenizer_model_name)
assert tok.eos_token_id is not None
eos = tok.eos_token_id

print(f"tokenizer:       {cfg.pretrained_tokenizer_model_name}")
print(f"tok.vocab_size:  {tok.vocab_size}")
print(f"len(tok):        {len(tok)}  (includes added/special tokens)")
print(f"eos_token / id:  {tok.eos_token!r} / {eos}")

tokenizer:       HuggingFaceTB/SmolLM2-135M
tok.vocab_size:  49152
len(tok):        49152  (includes added/special tokens)
eos_token / id:  '<|endoftext|>' / 0


## 1. Stream the dataset

`streaming=True` avoids the full ~28 GB download. Shuffle with a buffer before taking anything so the EDA sample isn't just the first shard.

In [4]:
ds_stream = load_dataset(dataset, name=dataset_config, split="train", streaming=True)
ds_stream = ds_stream.shuffle(seed=seed, buffer_size=10_000)
ds_stream

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
    num_shards: 14
})

In [5]:
first = next(iter(ds_stream))
print("fields:", list(first.keys()))
for k, v in first.items():
    s = repr(v)
    print(f"  {k:>14}: {s[:120]}{' ...' if len(s) > 120 else ''}")

fields: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score']
            text: "The entrance to Simbol Materials' Calipatria demonstration facility is seen on Wednesday, January 8, 2014 near a geothe ...
              id: '<urn:uuid:fc2c40a7-090c-409f-b6bb-a30830128e70>'
            dump: 'CC-MAIN-2015-22'
             url: 'http://archive.desertsun.com/article/20140222/BUSINESS0302/302220055/Simbol-Materials-lithium-extraction-Salton-Sea'
       file_path: 's3://commoncrawl/crawl-data/CC-MAIN-2015-22/segments/1432207924799.9/warc/CC-MAIN-20150521113204-00249-ip-10-180-206-21 ...
        language: 'en'
  language_score: 0.9487097859382629
     token_count: 3553
           score: 3.484375
       int_score: 3


## 2. Pull an EDA sample

Take `eda_sample` rows into memory for quick polars/plotly analysis. We retokenize each doc with our tokenizer so the reported lengths match the filter criterion used downstream — the dataset's native `token_count` is GPT-2-based and would misrepresent the filter threshold for other tokenizers.

In [6]:
rows = list(itertools.islice(ds_stream, eda_sample))
texts = [r["text"] for r in rows]
tok_counts = [
    len(tok.encode(t, add_special_tokens=False))
    for t in tqdm(texts, desc="retokenizing EDA sample")
]
eda = pl.DataFrame({
    "text": texts,
    "token_count": tok_counts,
    "gpt2_token_count": [r["token_count"] for r in rows],
    "language": [r.get("language") for r in rows],
    "score": [r.get("score") for r in rows],
    "url": [r.get("url") for r in rows],
})
eda = eda.with_columns(char_len=pl.col("text").str.len_chars())
eda.select("token_count", "gpt2_token_count", "char_len", "score").describe()

retokenizing EDA sample:   0%|          | 0/5000 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (11506 > 8192). Running this sequence through the model will result in indexing errors


statistic,token_count,gpt2_token_count,char_len,score
str,f64,f64,f64,f64
"""count""",5000.0,5000.0,5000.0,5000.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",987.7572,977.5694,4517.1934,3.008969
"""std""",1506.309342,1467.514084,6717.151477,0.4028
"""min""",68.0,67.0,243.0,2.515625
"""25%""",351.0,348.0,1636.0,2.6875
"""50%""",641.0,637.0,2981.0,2.90625
"""75%""",1078.0,1074.0,4945.0,3.265625
"""max""",30093.0,29204.0,128707.0,4.75


In [7]:
fig = px.histogram(
    {"token_count": eda["token_count"].to_list()},
    x="token_count",
    nbins=80,
    log_x=True,
    title=(
        f"FineWeb-Edu {dataset_config}: token_count ({eda_sample} docs, "
        f"{cfg.pretrained_tokenizer_model_name})"
    ),
)
fig.add_vline(
    x=cfg.sample_length,
    line_dash="dash",
    annotation_text=f"sample_length={cfg.sample_length}",
)
fig.show()

In [8]:
q = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
tc = eda["token_count"]
print(f"token_count quantiles ({cfg.pretrained_tokenizer_model_name}):")
for qi in q:
    print(f"  p{int(qi * 100):>2}: {tc.quantile(qi):>8.0f}")
keep = (tc >= cfg.sample_length).sum() / len(tc)
print(
    f"\n{keep:.1%} of docs have >= {cfg.sample_length} tokens — these are the candidates we keep."
)

token_count quantiles (HuggingFaceTB/SmolLM2-135M):
  p 1:      106
  p10:      213
  p25:      351
  p50:      641
  p75:     1078
  p90:     1861
  p99:     6936

97.9% of docs have >= 128 tokens — these are the candidates we keep.


## 3. Eyeball raw examples

One short doc, one medium, one long — so you can see the range of content that survives the filter.

In [9]:
picks = (
    eda
    .sort("token_count")
    .with_row_index()
    .filter(
        pl.col("index").is_in([
            len(eda) // 20,
            len(eda) // 2,
            len(eda) - len(eda) // 20,
        ])
    )
)
for row in picks.iter_rows(named=True):
    kept = row["token_count"] >= cfg.sample_length
    print(
        f"--- token_count={row['token_count']}  score={row['score']}  kept={kept} ---"
    )
    print(row["text"][:600].rstrip() + (" …" if len(row["text"]) > 600 else ""))
    print()

--- token_count=165  score=2.875  kept=True ---
'Air Walker': Electronics-Free, Air-Powered Robot
Engineers at the University of California San Diego have created a four-legged soft robot that doesn’t need any electronics to work; it only needs a constant source of pressurized air for all its functions, including its controls and locomotion systems. The UC San Diego bot is controlled by a lightweight, low-cost system of pneumatic circuits, made up of tubes and soft valves, onboard the robot itself. It can walk on command or in response to signals it senses from the environment. The robot is also equipped with simple mechanical sensors that …

--- token_count=641  score=2.828125  kept=True ---
14 Dec 2012
New Delhi, India.
Lower back pain – a common phenomenon among Indians – has been found to be the leading cause of years lived with disability (YLD) globally.
The Global Disease Burden study published on Thursday showed that lower back pain caused 83.1 million YLDs across the globe in 2

## 4. Filter + truncate to document prefixes

Stream, tokenize with `cfg.pretrained_tokenizer_model_name`, skip docs shorter than `cfg.sample_length`, take the first `cfg.sample_length` tokens of each kept doc. Collect `2 * cfg.num_samples` samples (train + val). Re-streams from scratch so the EDA sample above doesn't bias the final corpus.

In [10]:
target_total = 2 * cfg.num_samples

ds_collect = load_dataset(
    dataset, name=dataset_config, split="train", streaming=True
).shuffle(seed=seed, buffer_size=10_000)

samples: list[list[int]] = []
docs_seen = 0
docs_kept = 0
bar = tqdm(total=target_total, desc=f"collecting {cfg.sample_length}-token prefixes")

for ex in ds_collect:
    text = ex["text"].strip()
    if not text:
        continue
    docs_seen += 1
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) < cfg.sample_length:
        continue
    samples.append(ids[: cfg.sample_length])
    docs_kept += 1
    bar.update(1)
    if docs_kept >= target_total:
        break
bar.close()

keep_rate = docs_kept / max(docs_seen, 1)
print(f"kept {docs_kept} / {docs_seen} docs ({keep_rate:.1%} pass rate)")

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

collecting 128-token prefixes:   0%|          | 0/100000 [00:00<?, ?it/s]

kept 100000 / 102276 docs (97.8% pass rate)


## 5. Shuffle + shard into train/val

Docs arrive already shuffled (streaming buffer), but shuffle again for good measure so train/val are interchangeable. `polars.DataFrame.sample(fraction=1.0, shuffle=True, seed=...)` is deterministic given `seed`.

In [11]:
df = pl.DataFrame({"input_ids": samples}).sample(fraction=1.0, shuffle=True, seed=seed)
train_df = df.head(cfg.num_samples)
val_df = df.slice(cfg.num_samples, cfg.num_samples)
assert len(train_df) == cfg.num_samples and len(val_df) == cfg.num_samples
print(
    f"train: {len(train_df)}  val: {len(val_df)}  (sample_length={cfg.sample_length})"
)
train_df.head(2)

train: 50000  val: 50000  (sample_length=128)


input_ids
list[i64]
"[504, 1695, … 3251]"
"[20799, 284, … 3947]"


## 6. Inspect train samples

Decode a few prefixes back to text as a sanity check.

In [12]:
for i in (0, 1, 2, len(train_df) // 2, len(train_df) - 1):
    ids = train_df["input_ids"][i].to_list()
    print(f"--- train[{i}]  len={len(ids)} ---")
    print(tok.decode(ids))
    print()

--- train[0]  len=128 ---
The following biography is reprinted from Characteristics of English Poets from Chaucer to Shirley. William Minto. London: William Blackwood and Sons, 1885.
Within the first ten years of Elizabeth's reign a novelty was added to the drama. In 1566, George Gascoigne translated from Ariosto, for representation at Gray's Inn, the prose comedy Gli-Suppositi. This, acted under the title of The Supposes, is the first comedy written in English prose, and in plot, situation, and character, it approaches nearer than Damon and Pythias to the established

--- train[1]  len=128 ---
Climate and Weather Education | Kaplan Early Learning
Children are often scared of thunderstorms, but they love jumping in puddles of water and looking at rainbows after a storm ends. Making observations about the weather and discussing climate are great ways to teach children about science, especially since weather is something children can easily relate to and understand. Whether it's clear an

## 7. Persist

Two parquet shards under `out_dir`. Each row is one `input_ids` list of length `cfg.sample_length`.

In [13]:
train_df.write_parquet(out_train)
val_df.write_parquet(out_val)
for p in (out_train, out_val):
    print(f"wrote {p}  ({p.stat().st_size / 1e6:.1f} MB)")

wrote /home/nlyu/Code/maxrl-statistics/artifacts/corpus-regression/fineweb_edu_smollm2_135m_50000_x_128/train.parquet  (12.9 MB)
wrote /home/nlyu/Code/maxrl-statistics/artifacts/corpus-regression/fineweb_edu_smollm2_135m_50000_x_128/val.parquet  (12.9 MB)
